In [ ]:
rm(list=ls())
library(Seurat)
library(TOAST)
library(SingleCellExperiment)
library(Biobase)
library(dplyr)
library(org.Hs.eg.db)
library(MuSiC)
library(Matrix)
library(scuttle)
library(edgeR)
library(ggplot2)

In [ ]:
packageVersion("MuSiC")
packageVersion("ggplot2")

In [ ]:
base_path = '/home/EOCRC_atlas/'

In [ ]:
seurat_path = paste0(base_path, '/data/all_samples_raw_withTier2Annotation_09-19-25.rds')
save_path = paste0(base_path, '/results/2026_06_15_YOCRC_deconvolution/')
if (!dir.exists(save_path)) {dir.create(save_path, recursive = TRUE)}

In [ ]:
# Load Seurat object 
crc <- readRDS(seurat_path)

In [ ]:
# subset to just the cell types we want (all non-mixed marker cell types)
keep = c('Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
       'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
       'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
       'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
       'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
       'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
       'Glial cells', 'HSP-hi - B cell', 'HSP-hi Myeloid',
       'HSP-hi Stromal', 'HSP-hi T cells', 'HSP-hi glial', 'ILCs',
       'LGR5 stem cell-like', 'Lymphatic endothelium',
       'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
       'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
       'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
       'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
       'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
       'Regulatory T cells', 'T helper cells', 'Vascular endothelium')
crc = subset(crc, subset = Annotation_Tier2%in%keep)

In [ ]:
# subset to just MSS (and make sure we patients with known metadata for covariates of intesrest)
# sidedness and stage are over-kill, these variables are known for all MSS patients 
dim(crc)
crc = subset(crc, MSI_v2 == "MSS: STABLE" & Sidedness %in% c("Left", "Right", "Rectal") & Overall_Stage %in% c("I", "II", "III", "IV"))
dim(crc)

In [ ]:
# load in the TCGA-YOCRC bulk RNA-Seq object 
# the merged matrix is called "mat" 
load(paste0('/data/Filtered_TCGA_YOCRC_data.RData'))

In [ ]:
# convert bulk data to gene symbols 
clean_bulk_ids = gsub("\\..*", "", rownames(mat))
mapping_bulk = mapIds(org.Hs.eg.db, 
                       keys = clean_bulk_ids, 
                       column = "SYMBOL", 
                       keytype = "ENSEMBL", 
                       multiVals = "first")

valid_mask = !is.na(mapping_bulk)
mat_to_aggregate = mat[valid_mask, ]
symbols_for_agg = mapping_bulk[valid_mask]

print("Aggregating bulk counts by Gene Symbol...")
bulk_final = aggregate(mat_to_aggregate, 
                        by = list(symbols_for_agg), 
                        FUN = sum)

rownames(bulk_final) = bulk_final$Group.1
bulk_final = as.matrix(bulk_final[, -1])

print(dim(bulk_final))

In [ ]:
# Create an expression set from the bulk data 
bulk_eset = ExpressionSet(assayData = as.matrix(bulk_final))

In [ ]:
# identify genes to use in the deconvolution analysis 
# keep genes that are in common between the bulk and single cell datasets 
# keep genes that with average expression < 0.05 and aren't mito or ribo 
common_genes = intersect(rownames(crc), rownames(bulk_eset))
print(paste0(length(common_genes), " common genes"))

expressed <- AverageExpression(crc, group.by = "Annotation_Tier1", slot = "counts", use.scale = FALSE, use.counts = FALSE)$RNA
genes_minExpressed <- rownames(expressed)[apply(expressed, 1, max) > 0.05]

mito_genes = rownames(crc)[!grepl("^MT-|^RPS|^RPL", rownames(crc))]

gene_lists = list(common_genes, genes_minExpressed, mito_genes)
keep <- Reduce(intersect, gene_lists)

print(paste0(length(keep), " genes are kept after filtering"))

In [ ]:
# Filter bulk and single cell datasets to final gene set 
bulk_sub <- bulk_eset[keep,]
sc_sub <- crc[keep,]

In [ ]:
# Convert Seurat object into Single Cell Experiment Object 
counts_mtx <- GetAssayData(sc_sub, layer = "counts")
sc_sce <- SingleCellExperiment(assays = list(counts = counts_mtx), colData = sc_sub@meta.data)

In [ ]:
# Run MuSiC deconvolution 
decon_results <- music_prop(
    bulk.mtx = as.matrix(exprs(bulk_sub)),
    sc.sce = sc_sce,           
    clusters = 'Annotation_Tier1', 
    samples = 'FRID', 
    verbose = TRUE
)

In [ ]:
# Check median epithelial proportion 
median(decon_results$Est.prop.weighted[,'Epithelial'])

In [ ]:
# Create final matrix 
decon_df <- as.data.frame(decon_results$Est.prop.weighted)
decon_df$Barcode <- rownames(decon_df)

res <- merge(decon_df, meta, by = "Barcode")

In [ ]:
# check deconvolution accuracy 
sc_basis = aggregateAcrossCells(sc_sce, ids = sc_sce$Annotation_Tier1, statistics = "mean", use_altexps = FALSE)
basis_mtx = assay(sc_basis, 1)

common_genes = intersect(rownames(basis_mtx), rownames(bulk_sub))
basis_sub = basis_mtx[common_genes, ]
bulk_check_sub = exprs(bulk_sub)[common_genes, ]

res_sub = as.matrix(decon_results$Est.prop.weighted[, colnames(basis_sub)])
pred_bulk = basis_sub %*% t(res_sub)

bulk_log_cpm = log2(edgeR::cpm(bulk_check_sub) + 1)
pred_log_cpm = log2(edgeR::cpm(pred_bulk) + 1)

correlations = sapply(1:ncol(bulk_log_cpm), function(i) {cor(bulk_log_cpm[, i], pred_log_cpm[, i], method = "pearson")})
names(correlations) = colnames(bulk_log_cpm)
print(paste0("Median Deconvolution Accuracy (R): ", round(median(correlations), 4)))

In [ ]:
# add age scaled column 
res$age_scaled = scale(as.numeric(res$Age))

In [ ]:
# save results table 
saveRDS(res, paste0(save_path, 'music_deconvolution.Rds'))

In [ ]:
# Look at stromal to epi ratio in TCGA only 
res$log_epi_strom_ratio <- log(res$Stromal / res$Epithelial)
fit_log_ratio <- lm(log_epi_strom_ratio ~ age_scaled + Sex + Stage + Side + Treatment, data = res[res$Cohort=='TCGA' & is.finite(res$log_epi_strom_ratio), ])
summary(fit_log_ratio)

In [ ]:
# Look at stromal to epi ratio in TCGA + DFCI_EO-CRC 
res$log_epi_strom_ratio <- log(res$Stromal / res$Epithelial)
fit_log_ratio <- lm(log_epi_strom_ratio ~ age_scaled + Sex + Stage + Side + Cohort + Treatment, data = res[is.finite(res$log_epi_strom_ratio), ])
summary(fit_log_ratio)

In [ ]:
# Create plots of epi, stromal
plot_data = res
ggplot(plot_data, aes(x = Age, y = Epithelial)) +
  geom_point(aes(color = Cohort), alpha = 0.8, size = 1.5) +
  geom_smooth(method = "loess",               
              color = "black", 
              fill = "grey60", 
              linewidth = 1.2) +
  scale_color_brewer(palette = "Set1") +      
  theme_minimal() 
ggsave(paste0(save_path, 'deconvolution_tcga_dfci_epi.pdf'), width=4, height=4)

ggplot(plot_data, aes(x = Age, y = Stromal)) +
  geom_point(aes(color = Cohort), alpha = 0.8, size = 1.5) +
  geom_smooth(method = "loess",               
              color = "black", 
              fill = "grey60", 
              linewidth = 1.2) +
  scale_color_brewer(palette = "Set1") +     
  theme_minimal() 
ggsave(paste0(save_path, 'deconvolution_tcga_dfci_stromal.pdf'), width=4, height=4)

In [ ]:
# plot stromal to epi ratio 
ggplot(plot_data, aes(x = Age, y = log_epi_strom_ratio)) +
  geom_point(aes(color = Cohort), alpha = 0.8, size = 1.5) +
  geom_smooth(method = "loess",               
              color = "black", 
              fill = "grey60", 
              linewidth = 1.2) +
  scale_color_brewer(palette = "Set1") +     
  theme_minimal()  
ggsave(paste0(save_path, 'deconvolution_tcga_dfci_stromal_epi_ratio.pdf'), width=4, height=4)